In [ ]:
import os

# 1. Deteksi environment (Colab vs Lokal/VS Code)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    base_dir = "/content"
else:
    base_dir = os.getcwd()

# 2. Clone atau Sinkronisasi Repo Github
repo_url = "https://github.com/gacoanphilia/Document-Image-Classification.git"
repo_dir = os.path.join(base_dir, "Document-Image-Classification")

if not os.path.exists(repo_dir):
    print("Mengunduh repo baru (Clone)...")
    !git clone {repo_url}
else:
    print("Repo ditemukan, menyinkronkan pembaruan (Pull)...")
    %cd {repo_dir}
    !git pull origin main

%cd {repo_dir}
!mkdir -p data

# 3. Ekstrak data jika berada di Colab
if IN_COLAB:
    zip_path = "/content/drive/MyDrive/Semester 5/SD4504_SML/Document-Image-Classification/data/document-image-classification-sd234504.zip"
    print("Mengekstrak ZIP ke folder 'data/'...")
    !unzip -q -o "{zip_path}" -d data/
else:
    print("Eksekusi Lokal: Pastikan file test.csv dan folder test/ berada di dalam folder 'data/'.")

print("Setup Environment Selesai!")

In [ ]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b0
from PIL import Image
from tqdm import tqdm

# Hardware Routing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device untuk Inferensi: {device}")

# Path Setup
DATA_DIR = Path("data").resolve()
TEST_CSV = DATA_DIR / "test.csv"
TEST_IMG_DIR = DATA_DIR / "test"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"

# Path model dari 02_modeling (Pencarian fleksibel di folder Colab / Lokal)
MODEL_PATH = Path("..").resolve() / "models" / "best_model.pth"
if not MODEL_PATH.exists():
    MODEL_PATH = Path("models") / "best_model.pth"
if not MODEL_PATH.exists():
    MODEL_PATH = Path("/content/models/best_model.pth")

In [ ]:
# Transformasi khusus inferensi (Konsisten dengan val_transform pada 02_modeling)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class DocumentTestDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.transform = transform
        
        # Deteksi kolom nama file gambar secara otomatis
        if 'filename' in self.df.columns:
            self.file_col = 'filename'
        elif 'id' in self.df.columns:
            self.file_col = 'id'
        else:
            self.file_col = self.df.columns[0]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = str(self.df.iloc[idx][self.file_col])
        img_path = self.img_dir / img_name
        
        # Penanganan opsional jika ekstensi file tidak tertulis di CSV
        if not img_path.exists():
            for ext in ['.jpg', '.png', '.tif', '.jpeg']:
                if (self.img_dir / f"{img_name}{ext}").exists():
                    img_path = self.img_dir / f"{img_name}{ext}"
                    break

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
            
        return image, img_name

test_dataset = DocumentTestDataset(csv_file=TEST_CSV, img_dir=TEST_IMG_DIR, transform=test_transform)
test_loader = DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print(f"Total gambar data test dimuat: {len(test_dataset)} sampel ({len(test_loader)} batch)")

In [ ]:
NUM_CLASSES = 16

# Inisialisasi Arsitektur EfficientNet-B0
model = efficientnet_b0(weights=None)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

# Memuat Trained Checkpoint
if MODEL_PATH.exists():
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"Berhasil memuat bobot dari: {MODEL_PATH}")
else:
    raise FileNotFoundError(f"File '{MODEL_PATH}' tidak ditemukan! Selesaikan training di 02_modeling.ipynb terlebih dahulu.")

model = model.to(device)
model.eval() # Matikan Dropout dan Batch Normalization update

In [ ]:
all_predictions = []
all_img_names = []

print("Memulai proses inferensi pada data test...")
with torch.no_grad():
    for images, img_names in tqdm(test_loader, desc="Inference"):
        images = images.to(device)
        outputs = model(images)
        
        # Ambil indeks kelas dengan logits tertinggi
        _, preds = torch.max(outputs, 1)
        
        all_predictions.extend(preds.cpu().numpy())
        all_img_names.extend(img_names)

print("Proses inferensi selesai!")

In [ ]:
# Membaca format bawaan sample_submission.csv
sub_df = pd.read_csv(SAMPLE_SUB)

id_col = sub_df.columns[0]
label_col = sub_df.columns[1] if len(sub_df.columns) > 1 else 'target'

# Pemetaan hasil prediksi berdasarkan nama gambar
pred_map = dict(zip(all_img_names, all_predictions))

if id_col in sub_df.columns:
    sub_df[label_col] = sub_df[id_col].astype(str).map(pred_map).fillna(all_predictions)
else:
    sub_df[label_col] = all_predictions

# Simpan ke CSV
OUTPUT_SUB_PATH = Path("submission.csv")
sub_df.to_csv(OUTPUT_SUB_PATH, index=False)

print(f"File submission berhasil diekspor ke: {OUTPUT_SUB_PATH.resolve()}")
print("\nPreview 10 baris pertama submission:")
print(sub_df.head(10))

print("\nDistribusi Prediksi Kelas:")
print(sub_df[label_col].value_counts().sort_index())